# Data Cleaning and feature engineering


In [1]:
import pandas as pd

In [2]:
pd.set_option('display.max_colwidth', None) # to increase the width of the columns

In [3]:
# orders.csv
url = "https://drive.google.com/file/d/1Vu0q91qZw6lqhIqbjoXYvYAQTmVHh6uZ/view?usp=sharing"
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
orders = pd.read_csv(path)

# orderlines.csv
url = "https://drive.google.com/file/d/1FYhN_2AzTBFuWcfHaRuKcuCE6CWXsWtG/view?usp=sharing"
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
orderlines = pd.read_csv(path)

In [5]:
#Copy
orders_df = orders.copy()

In [6]:
#Copy
orderlines_df = orderlines.copy()

## 1. Checking for duplicates


In [ ]:
# orders
orders_df.duplicated().sum()

0

In [ ]:
# orderlines
orderlines_df.duplicated().sum()

0

So far I have no duplicate rows in either DataFrame. There is no problem to solve or row to drop

# 2. Checking data types

In [ ]:
orders_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 226909 entries, 0 to 226908
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   order_id      226909 non-null  int64  
 1   created_date  226909 non-null  object 
 2   total_paid    226904 non-null  float64
 3   state         226909 non-null  object 
dtypes: float64(1), int64(1), object(2)
memory usage: 6.9+ MB


# NOTE:
* `total_paid` has 5 missing values
* `created_date` should become datetime datatype

In [ ]:
orderlines_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 293983 entries, 0 to 293982
Data columns (total 7 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   id                293983 non-null  int64 
 1   id_order          293983 non-null  int64 
 2   product_id        293983 non-null  int64 
 3   product_quantity  293983 non-null  int64 
 4   sku               293983 non-null  object
 5   unit_price        293983 non-null  object
 6   date              293983 non-null  object
dtypes: int64(4), object(3)
memory usage: 15.7+ MB


# NOTE:
* `date` should be a datetime datatype
* `unit_price` should be a float datatype
* `orderlines_df` has no missing values


## 3.&nbsp; Missing values

### 3.1. Orders table
* `total_paid` has 5 missing values

In [ ]:
#Percentage of missing values

print(f"5 missing values represents {((orders_df.total_paid.isna().sum() / orders_df.shape[0])*100).round(5)}% of the rows in our DataFrame")

5 missing values represents 0.0022% of the rows in our DataFrame


In [ ]:
orders_df.total_paid.isna().value_counts(normalize=True)

False    0.999978
True     0.000022
Name: total_paid, dtype: float64

As there is such a tiny amount of missing values, I will simply delete these rows, there is enough data without them to work with.

In [ ]:
#Deleting the missing values in the column total_paid

orders_df = orders_df.loc[~orders.total_paid.isna(), :]

### 3.2.&nbsp; Orderlines
There are no missing values in `orderlines`

## 4.&nbsp; Datatypes

### 4.1.&nbsp; Orders
* `created_date` should become datetime datatype

In [ ]:
#Transforming "created_date" to a datetime format, ready for analysis

orders_df["created_date"] = pd.to_datetime(orders_df["created_date"])

### 4.1.&nbsp; Orderlines
* `date` should be a datetime datatype
* `unit_price` should be a float datatype

#### 4.1.1.&nbsp; `date`

In [ ]:
#Transforming "date" to a datetime format, ready for analysis

orderlines_df["date"] = pd.to_datetime(orderlines_df["date"])

#### 4.1.2.&nbsp;`unit_price`

In [7]:
#Transforming "unit_price" to a float datatype. Using "pd.to_numeric"

orderlines_df["unit_price"] = pd.to_numeric(orderlines_df["unit_price"])

ValueError: Unable to parse string "1.137.99" at position 6

In trying to convert `unit_price` to a numerical datatype, I get a `ValueError` telling me that pandas doesn't understand the number `1.137.99`. This is probably because numbers cannot have 2 decimal points.

Let me check if there are any other numbers like this.

In [8]:
# Checking for problematic values

orderlines_df["unit_price"].unique()

array(['18.99', '399.00', '474.05', ..., '132.51', '3.475.00', '48.68'],
      dtype=object)

In [9]:
#Checking affected rows

orderlines_df[
    orderlines_df["unit_price"].str.contains(r"\.", regex=True)
]

# orderlines_df[
    #orderlines_df.unit_price.str.contains(r"\d+\.\d+\.\d+", regex=True)
#]

,id,id_order,product_id,product_quantity,sku,unit_price,date
0,1119109,299539,0,1,OTT0133,18.99,2017-01-01 00:07:19
1,1119110,299540,0,1,LGE0043,399.00,2017-01-01 00:19:45
2,1119111,299541,0,1,PAR0071,474.05,2017-01-01 00:20:57
3,1119112,299542,0,1,WDT0315,68.39,2017-01-01 00:51:40
4,1119113,299543,0,1,JBL0104,23.74,2017-01-01 01:06:38
...,...,...,...,...,...,...,...
293978,1650199,527398,0,1,JBL0122,42.99,2018-03-14 13:57:25
293979,1650200,527399,0,1,PAC0653,141.58,2018-03-14 13:57:34
293980,1650201,527400,0,2,APP0698,9.99,2018-03-14 13:57:41
293981,1650202,527388,0,1,BEZ0204,19.99,2018-03-14 13:58:01


In [11]:
#Checking affected rows

orderlines_df.unit_price.str.contains(r"\d+\.\d+\.\d+").value_counts()

#NOTE: Over 36,000 rows in orderlines are affected by this problem

,count
unit_price,
False,257814
True,36169


#How much that is as a percentage of the total data?

In [14]:
#How much that is as a percentage of the total data.

two_dot_percentage = ((orderlines_df.unit_price.str.contains(r"\d+\.\d+\.\d+").value_counts()[1] / orderlines_df.shape[0])*100).round(2)
print(f"The 2 dot problem represents {two_dot_percentage}% of the rows in our DataFrame")

The 2 dot problem represents 12.3% of the rows in our DataFrame


/tmp/ipykernel_445/862200048.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  two_dot_percentage = ((orderlines_df.unit_price.str.contains(r"\d+\.\d+\.\d+").value_counts()[1] / orderlines_df.shape[0])*100).round(2)


This is a bit of a tricky decision as 12.3% is a significant amount of the data... and I might even end up losing a larger portion of the data than this too. For the moment I will delete the rows as I only have 2 weeks for this project and I'd like some quick, accurate results to show. If I have time at the end, I can come back and investigate this problem further, maybe there's a solution?

Each row of `orderlines` represents a product in an order. For example, if order number 175 contained 3 seperate products, then order 175 would have 3 rows in `orderlines`, one row for each of the products. If 2 of those products have 'normal' prices (14.99, 15.85) and 1 has a price with 2 decimal points (1.137.99), I will need to remove the whole order and not just the affected row. If I only remove the row with 2 decimal places then any later analysis about products and prices could be misleading.

I therefore need to find the order numbers associated with the rows that have 2 decimal points, and then remove all the associated rows.

In [ ]:
two_dot_order_ids_list = orderlines_df.loc[orderlines_df.unit_price.str.contains(r"\d+\.\d+\.\d+"), "id_order"]

orderlines_df = orderlines_df.loc[~orderlines_df.id_order.isin(two_dot_order_ids_list)]

In [ ]:
orderlines_df.shape[0]

216250

We still have 216250 rows in orderlines to work with. This should be more than enough for our evaluation.

Now that all of the 2 decimal point prices have been removed, let's try again to convert the column `unit_price` to the correct datatype.

In [ ]:
#Now converting "unit_price" to the correct datatype

orderlines_df["unit_price"] = pd.to_numeric(orderlines_df["unit_price"])

# Cleaning  the "products" DataFrame


In [15]:
import pandas as pd

# products.csv
url = "https://drive.google.com/file/d/1KPSQ0kxkJq8L309FSaslqY2e-lvTPsMX/view?usp=drive_link"
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
products = pd.read_csv(path)

In [16]:
products_df = products.copy()

In [17]:
products_df.head()

,sku,name,desc,price,promo_price,in_stock,type
0,RAI0007,Silver Rain Design mStand Support,Aluminum support compatible with all MacBook,59.99,499.899,1,8696
1,APP0023,Apple Mac Keyboard Keypad Spanish,USB ultrathin keyboard Apple Mac Spanish.,59,589.996,0,13855401
2,APP0025,Mighty Mouse Apple Mouse for Mac,mouse Apple USB cable.,59,569.898,0,1387
3,APP0072,Apple Dock to USB Cable iPhone and iPod white,IPhone dock and USB Cable Apple iPod.,25,229.997,0,1230
4,KIN0007,Mac Memory Kingston 2GB 667MHz DDR2 SO-DIMM,2GB RAM Mac mini and iMac (2006/07) MacBook Pro (2006/07/08).,34.99,31.99,1,1364


In [ ]:
products_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19326 entries, 0 to 19325
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   sku          19326 non-null  object
 1   name         19326 non-null  object
 2   desc         19319 non-null  object
 3   price        19280 non-null  object
 4   promo_price  19326 non-null  object
 5   in_stock     19326 non-null  int64 
 6   type         19276 non-null  object
dtypes: int64(1), object(6)
memory usage: 1.0+ MB


We'll go through the steps above in order
* Duplicates
* Missing values
* Datatypes

But I think we can all see straight away from `products.head()` above that some of the prices in `promo_price` look wrong. Let's make sure we deal with this later.

## Duplicates

In [18]:
products_df.duplicated().sum()

np.int64(8746)

That's a lot of duplicates. This DataFrame should contain one product per line, so duplicate data doesn't make sense. I'll have to drop them.

In [19]:
#Dropping duplicates

products_df = products_df.drop_duplicates().copy()

In [20]:
products_df

,sku,name,desc,price,promo_price,in_stock,type
0,RAI0007,Silver Rain Design mStand Support,Aluminum support compatible with all MacBook,59.99,499.899,1,8696
1,APP0023,Apple Mac Keyboard Keypad Spanish,USB ultrathin keyboard Apple Mac Spanish.,59,589.996,0,13855401
2,APP0025,Mighty Mouse Apple Mouse for Mac,mouse Apple USB cable.,59,569.898,0,1387
3,APP0072,Apple Dock to USB Cable iPhone and iPod white,IPhone dock and USB Cable Apple iPod.,25,229.997,0,1230
4,KIN0007,Mac Memory Kingston 2GB 667MHz DDR2 SO-DIMM,2GB RAM Mac mini and iMac (2006/07) MacBook Pro (2006/07/08).,34.99,31.99,1,1364
...,...,...,...,...,...,...,...
19321,BEL0376,Belkin Travel Support Apple Watch Black,compact and portable stand vertically or horizontally for Apple Watch,29.99,269.903,1,12282
19322,THU0060,"Enroute Thule 14L Backpack MacBook 13 ""Black",Backpack with capacity of 14 liter compartments MacBook up to 13 inches up to 10 inches Cases,69.95,649.903,1,1392
19323,THU0061,"Enroute Thule 14L Backpack MacBook 13 ""Blue",Backpack with capacity of 14 liter compartments MacBook up to 13 inches up to 10 inches Cases,69.95,649.903,1,1392
19324,THU0062,"Enroute Thule 14L Backpack MacBook 13 ""Red",Backpack with capacity of 14 liter compartments MacBook up to 13 inches up to 10 inches Cases,69.95,649.903,0,1392


In [ ]:
#Check

products_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10580 entries, 0 to 19325
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   sku          10580 non-null  object
 1   name         10580 non-null  object
 2   desc         10573 non-null  object
 3   price        10534 non-null  object
 4   promo_price  10580 non-null  object
 5   in_stock     10580 non-null  int64 
 6   type         10530 non-null  object
dtypes: int64(1), object(6)
memory usage: 661.2+ KB


# NOTES:
* `price` and `desc` have missing values
* `type` also has some missing values

* `price` and `promo_price`have been stored as objects and not as a numerical datatypes. This needs to be fixed.

### Missing values
I can see from `.info()` above that there are missing values in `desc` and `price`. Let me check how many they are.

In [ ]:
#Checking the number of missing values in column "desc"

products_df["desc"].isna().sum()

np.int64(7)

In [ ]:
#Printing the rows that have missing values

products_df.loc[products_df['desc'].isna(), :]

,sku,name,desc,price,promo_price,in_stock,type
16126,WDT0211-A,"Open - Purple 2TB WD 35 ""PC Security Mac hard ...",NaN,107,814.659,0,1298
16128,APP1622-A,Open - Apple Smart Keyboard Pro Keyboard Folio...,NaN,1.568.206,1.568.206,0,1298
17843,PAC2334,Synology DS718 + NAS Server | 10GB RAM,NaN,566.35,5.659.896,0,12175397
18152,KAN0034-A,Open - Kanex USB-C Gigabit Ethernet Adapter Ma...,NaN,29.99,237.925,0,1298
18490,HTE0025,Hyper Pearl 1600mAh battery Mini USB Mirror an...,NaN,24.99,22.99,1,1515
18612,OTT0200,OtterBox External Battery Power Pack 20000 mAHr,NaN,79.99,56.99,1,1515
18690,HOW0001-A,Open - Honeywell thermostat Lyric zonificador ...,NaN,199.99,1.441.174,0,11905404


# NOTES:
* `price` and `promo_price`also have the 2 decimal problem as in the orderlines_df table before. There are also prices with 3 decimal places.
* The product `names` for the missing `desc` here are quite descriptive

In [ ]:
# The products "names" here are quite descriptive, so I'll attempt to just copy
# them to the "desc" column, so that there is a description in case I need it later.


products_df.loc[products_df['desc'].isna(), 'desc'] = products_df.loc[products_df['desc'].isna(), 'name']

In [ ]:
#Running the code again to make sure it worked

products_df.loc[products_df['desc'].isna(), :]

,sku,name,desc,price,promo_price,in_stock,type


Adressing  the `price` and `promo_price` columns.

#### `price`

In [ ]:
#Price column
products_df.price.isna().sum()

np.int64(46)

In [22]:
#Promo_price column
products_df.promo_price.isna().sum()

np.int64(0)

In [ ]:
#Percentage of missing values

print(f"The missing values in price are {(products_df.price.isna().value_counts(normalize=True)[True] * 100).round(2)}% of all rows in the DataFrame")

The missing values in price are 0.43% of all rows in the DataFrame


In [21]:
#Percentage of missing values

products_df.price.isna().value_counts(normalize=True)

,proportion
price,
False,0.995652
True,0.004348


In [ ]:
#Delete rows with the missing values to ensure trust in the numbers in the final DataFrame

## Price is very important when investigating discounts.

products_df = products_df.dropna(subset=['price']).copy()

# or
## products_df = products_df.loc[~products['price'].isna()].copy()

# `type` also has some missing values

In [ ]:
products_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10534 entries, 0 to 19325
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   sku          10534 non-null  object
 1   name         10534 non-null  object
 2   desc         10534 non-null  object
 3   price        10534 non-null  object
 4   promo_price  10534 non-null  object
 5   in_stock     10534 non-null  int64 
 6   type         10484 non-null  object
dtypes: int64(1), object(6)
memory usage: 658.4+ KB


Type isn’t an essential piece of data for the analysis and is therefore allowed to carry missing values.
The only place it comes in later is as an optional route to category creation, where rows with missing values can still be dropped. However one can still use name and desc to categorize those rows.


### Data types

In the output of `.info()` above, both `price` and `promo_price` have been stored as objects and not as a numerical datatypes. Also both columns have some prices with 3 decimal places and others with 2 decimal points - the latter will prevent from converting the datatype to numerical, it must investigate and solved.

#### `price`

Number of values affected by the 2-decimal-dot problems or 3 decimal places.

In [ ]:
#two_dots
two_dots = products_df["price"].str.count(r"\.") > 1 # is there more than period?

#three_dots
three_decimals = products_df["price"].str.contains(r"\d+\.\d{3,}$") # does the string have one or more numbers, followed by a period, followed by at least 3 numbers at the end of the string

price_problems_number = products_df.loc[two_dots | three_decimals].shape[0]

price_problems_number

542

In [ ]:
print(f"The column price has in total {price_problems_number} wrong values. This is {round(((price_problems_number / products_df.shape[0]) * 100), 2)}% of the rows of the DataFrame")

The column price has in total 542 wrong values. This is 5.15% of the rows of the DataFrame


5.15% is a reasonable amount of the data. However, the price column will be important to understanding discounts, so I'd like it to be very trustworthy as business decisions are based on it. Therefore, I'll delete these rows

In [ ]:
#Deleting the affected row. Keeping the data reliable

products_df = products_df.loc[~(two_dots | three_decimals)].copy()

In [ ]:
products_df

,sku,name,desc,price,promo_price,in_stock,type
0,RAI0007,Silver Rain Design mStand Support,Aluminum support compatible with all MacBook,59.99,499.899,1,8696
1,APP0023,Apple Mac Keyboard Keypad Spanish,USB ultrathin keyboard Apple Mac Spanish.,59,589.996,0,13855401
2,APP0025,Mighty Mouse Apple Mouse for Mac,mouse Apple USB cable.,59,569.898,0,1387
3,APP0072,Apple Dock to USB Cable iPhone and iPod white,IPhone dock and USB Cable Apple iPod.,25,229.997,0,1230
4,KIN0007,Mac Memory Kingston 2GB 667MHz DDR2 SO-DIMM,2GB RAM Mac mini and iMac (2006/07) MacBook Pr...,34.99,31.99,1,1364
...,...,...,...,...,...,...,...
19321,BEL0376,Belkin Travel Support Apple Watch Black,compact and portable stand vertically or horiz...,29.99,269.903,1,12282
19322,THU0060,"Enroute Thule 14L Backpack MacBook 13 ""Black",Backpack with capacity of 14 liter compartment...,69.95,649.903,1,1392
19323,THU0061,"Enroute Thule 14L Backpack MacBook 13 ""Blue",Backpack with capacity of 14 liter compartment...,69.95,649.903,1,1392
19324,THU0062,"Enroute Thule 14L Backpack MacBook 13 ""Red",Backpack with capacity of 14 liter compartment...,69.95,649.903,0,1392


Now converting the column to a numeric datatype

In [ ]:
products_df["price"] = pd.to_numeric(products_df["price"])

In [ ]:
products_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9992 entries, 0 to 19325
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   sku          9992 non-null   object 
 1   name         9992 non-null   object 
 2   desc         9992 non-null   object 
 3   price        9992 non-null   float64
 4   promo_price  9992 non-null   object 
 5   in_stock     9992 non-null   int64  
 6   type         9946 non-null   object 
dtypes: float64(1), int64(1), object(5)
memory usage: 624.5+ KB


#### `promo_price`

Same steps: Number of values affected by the 2-decimal-dots problem, or the 3 decimal-places problem

In [ ]:
#two_dots_promo
two_dots_promo = products_df["promo_price"].str.count(r"\.") > 1 # is there more than period?

#three_decimals_promo
three_decimals_promo = products_df["promo_price"].str.contains(r"\d+\.\d{3,}") # does the string have one or more numbers, followed by a period, followed by at least 3 numbers at the end of the string

promo_problems_number = products_df.loc[two_dots_promo | three_decimals_promo].shape[0]

promo_problems_number

9232

In [ ]:
print(f"The column promo_price has in total {promo_problems_number} wrong values. This is {round(((promo_problems_number / products_df.shape[0]) * 100), 2)}% of the rows of the DataFrame")

The column promo_price has in total 9232 wrong values. This is 92.39% of the rows of the DataFrame


WOW!!! That's a lot of wrong data. The column `promo_price` has in total 9,232 wrong values. This is *92.39%* of the rows in the DataFrame.

Netx, I'll sample the data to check that all the numbers in the "promo_price" column really have either 2 decimal points or 3 decimal places.

In [ ]:
#Sampling the data to check that all the numbers in the "promo_price" column really have either 2 decimal points or 3 decimal places.

promo_price_df = products_df.loc[two_dots_promo | three_decimals_promo]
promo_price_df.sample(50)

,sku,name,desc,price,promo_price,in_stock,type
1813,SAM0073,Samsung 850 EVO SSD Disk 250GB,SSD hard drive Mac and PC 25 inch 250GB SATA I...,107.99,869.942,1,12215397
14680,PAC1756,QNAP TS-231P NAS,NAS with 20TB capacity Seagate Hard Drives for...,1416.57,9.401.785,0,12175397
14979,BNQ0046-A,Open - BenQ EW2775ZH VA Panel Monitor 27 Slim ...,Monitor slim frame 27 Intelligent Brightness t...,219.99,1.895.239,0,1298
11315,KIN0148,Kingston SDXC Memory Card UHS Class 3 | 128 GB,SDXC Memory Card UHS Class 3 128GB with speeds...,59.99,582.809,0,57445397
15606,APP1978,Apple iPad Wi-Fi 32GB Space Gray,New iPad Wi-Fi 32GB Space Gray (MP2F2TY / A),402.81,3.910.018,1,1714
11528,SAN0115,"IXpand SanDisk Flash Drive USB 2.0 128GB, and ...",USB 2.0 Pen Drive and Lightning 16GB for iPad ...,149.99,999.896,0,42945397
17160,AP20237,"Like new - Apple MacBook Air 13 ""Core i7 22GHz...",Air reconditioned computer MacBook 13 inch i7 ...,1219.00,975.594,0,"2,17E+11"
12007,MDT0018,Mediterrans Case iPhone 6 / 6S Azul Costa Brava,Matte finish Case for iPhone 6 / 6S.,19.00,169.896,0,11865403
14633,WDT0366,"4TB WD My Passport External Hard Disk 25 ""USB ...",4TB External Hard Drive USB 3.0 with Mac and P...,179.99,136.995,1,11935397
16663,APP2142,"Apple iPad Pro 10.5 ""Wi-Fi + Cellular 512GB Si...",Pro New iPad Wi-Fi + Cellular 512GB,1279.00,12.390.013,0,106431714


NOTE: Over 90% of the data in this column is corrupt. There's no point deleting all of these rows. Deleting will mean that there is barely a products table to work with later.

Instead, as it's only this column that appears to be very untrustworthy, I will delete the column.

In [ ]:
#Dropping the column "promo_price"
products_cl = products_df.drop("promo_price", axis=1)

In [ ]:
products_cl.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9992 entries, 0 to 19325
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   sku       9992 non-null   object 
 1   name      9992 non-null   object 
 2   desc      9992 non-null   object 
 3   price     9992 non-null   float64
 4   in_stock  9992 non-null   int64  
 5   type      9946 non-null   object 
dtypes: float64(1), int64(1), object(4)
memory usage: 546.4+ KB


Obviously, there's now no need to convert `promo_price` to a numerical datatype.


#Saving the new cleaned versions of the DataFrames.

In [ ]:
from google.colab import files # only when working on Colab!!

products_cl.to_csv("products_cl.csv", index=False)

files.download("products_cl.csv") # this downloads the file automatically